In [1]:
!pip install chromadb sentence-transformers transformers datasets accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.3 MB/s eta

In [2]:
pip install chromadb

In [3]:
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
import torch
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


In [4]:
ds = load_dataset("Amod/mental_health_counseling_conversations", split="train")
df = ds.to_pandas().head(500)
df


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...
...,...,...
495,I've hit my head on walls and floors ever sinc...,The best way to handle anxiety of this level i...
496,I have several issues that I need to work thro...,I am sorry that you had this experience. Thera...
497,Why am I so afraid of it? I don't understand.,Your fear is somewhat reasonable. No one want...
498,Why am I so afraid of it? I don't understand.,Why are you afraid of rape? Because it is a pr...


In [5]:
train_df = df.sample(frac=0.7, random_state=42)
test_df = df.drop(train_df.index)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


Train size: 350
Test size: 150


In [6]:
import os
if "COLAB_GPU" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()

In [ ]:
import os
from huggingface_hub import notebook_login

# If running in Google Colab
if "COLAB_GPU" in os.environ:
    !huggingface-cli login
# If running locally (Jupyter, VS Code, etc.)
else:
    notebook_login() 


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: read).
The token `lk` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no 

In [8]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
model_name = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)



tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [10]:
print("Tokenizer max length:", tokenizer.model_max_length)
print("Model max positions:", model.config.max_position_embeddings)

Tokenizer max length: 1000000000000000019884624838656
Model max positions: 32768


In [11]:
def count_tokens(text):
    return len(tokenizer.encode(text))

In [12]:
df["context_tokens"]  = df["Context"].apply(count_tokens)
df["response_tokens"] = df["Response"].apply(count_tokens)
df["combined_tokens"] = (df["Context"] + " " + df["Response"]).apply(count_tokens)

df


,Context,Response,context_tokens,response_tokens,combined_tokens
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb...",81,204,284
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",81,460,540
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...,81,69,149
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...,81,169,248
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,81,67,147
...,...,...,...,...,...
495,I've hit my head on walls and floors ever sinc...,The best way to handle anxiety of this level i...,85,218,302
496,I have several issues that I need to work thro...,I am sorry that you had this experience. Thera...,62,606,667
497,Why am I so afraid of it? I don't understand.,Your fear is somewhat reasonable. No one want...,15,265,279
498,Why am I so afraid of it? I don't understand.,Why are you afraid of rape? Because it is a pr...,15,344,358


In [13]:
df.describe()

,context_tokens,response_tokens,combined_tokens
count,500.00000,500.000000,500.000000
mean,68.18400,212.732000,279.868000
std,44.34726,144.182848,155.059425
min,8.00000,3.000000,38.000000
25%,36.00000,115.000000,169.750000
50%,64.00000,177.000000,239.000000
75%,92.25000,267.250000,342.000000
max,377.00000,901.000000,925.000000


In [ ]:
# CREATE CHROMA COLLECTION

import chromadb
from chromadb.config import Settings

# Use PersistentClient for local persistence
client = chromadb.PersistentClient(path="./chroma_db04")

collection = client.create_collection(
    name="mental_health_rags",
    metadata={"hnsw:space": "cosine"}
)

In [15]:
ids = []
documents = []
metadatas = []

for i, row in train_df.iterrows():
    ctx = row["Context"]
    resp = row["Response"]

    ids.append(str(i))
    documents.append(ctx)   # vectorized text = ONLY context

    metadatas.append({
        "context": ctx,
        "actual_response": resp,
        "row_index": i
    })


In [16]:
embeddings = embed_model.encode(documents, convert_to_numpy=True)

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)


In [21]:
def build_rag_prompt(user_query, retrieved_contexts, retrieved_responses):
    # Build example block
    examples = ""
    for i, (ctx, resp) in enumerate(zip(retrieved_contexts, retrieved_responses), start=1):
        examples += f"""
Example {i}
User: {ctx}
Assistant: {resp}
"""

    prompt = f"""
Below are examples of how a counselor responds with empathy and support.
Use these examples only as style guidance.
Follow the tone and style closely.
You may reuse similar words and phrases when appropriate.

{examples}

Now the user says:
User: {user_query}

Assistant:"""
    return prompt


In [22]:
results = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="RAG Testing"):

    test_query = row["Context"]
    actual_test_response = row["Response"]

    # embed + retrieve
    q_emb = embed_model.encode([test_query], convert_to_numpy=True).tolist()
    retrieved = collection.query(query_embeddings=q_emb, n_results=3)

    retrieved_contexts  = retrieved["documents"][0]
    retrieved_responses = [m["actual_response"] for m in retrieved["metadatas"][0]]

    # build prompt
    prompt = build_rag_prompt(
        test_query,
        retrieved_contexts,
        retrieved_responses
    )

    # generate output
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    gen_output = tokenizer.decode(
        out_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    results.append({
        "test_index": idx,
        "user_query": test_query,
        "actual_test_response": actual_test_response,
        "retrieved_train_responses": retrieved_responses,
        "rag_generated_output": gen_output
    })


RAG Testing:   0%|          | 0/150 [00:00<?, ?it/s]

In [23]:
evaluation_df = pd.DataFrame(results)
evaluation_df


,test_index,user_query,actual_test_response,retrieved_train_responses,rag_generated_output
0,1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",[Feelings of worthlessness often originate fr...,It sounds like you are experiencing significa...
1,4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,[Feelings of worthlessness often originate fr...,I understand you’re struggling with a lot rig...
2,8,I'm going through some things with my feelings...,It sounds like you may be putting yourself las...,[Feelings of worthlessness often originate fr...,I understand that you're struggling with feel...
3,13,I'm going through some things with my feelings...,I'm glad you are interested in changing your f...,[Feelings of worthlessness often originate fr...,It sounds like you are experiencing a signifi...
4,14,I'm going through some things with my feelings...,You have\nseveral things going on here. The sl...,[Feelings of worthlessness often originate fr...,It sounds like you are having some serious ch...
...,...,...,...,...,...
145,486,I have a fear of something and I want to face ...,This answer could be very different depending ...,"[Fear is a part of life. In fact, our five mai...",It's completely normal to feel uncertain abou...
146,488,I have a fear of something and I want to face ...,"Hello, and thank you for your question. Overco...","[Fear is a part of life. In fact, our five mai...",It's good that you're taking this step! It's ...
147,492,I don't remember when the voices in my head st...,This isn't something you can do on your own. I...,[You are right. It is not normal to hear voice...,I understand that hearing voices can be incre...
148,496,I have several issues that I need to work thro...,I am sorry that you had this experience. Thera...,[Despite your anxiety you are highly attuned t...,I hear you. It sounds like you've experienced...


In [25]:
!pip install rouge-score bert-score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00


In [27]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
!pip install rouge-score bert-score -q

Rouge Score

In [29]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import pandas as pd


In [28]:
from rouge_score import rouge_scorer
import pandas as pd
import evaluate

preds = evaluation_df["rag_generated_output"].tolist()
refs  = evaluation_df["actual_test_response"].tolist()

rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=preds,
    references=refs,
    rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"]
)

print("ROUGE Scores:")
print(rouge_scores)



ROUGE Scores:
{'rouge1': np.float64(0.29072590809573395), 'rouge2': np.float64(0.04117799124218667), 'rougeL': np.float64(0.13796448820445223), 'rougeLsum': np.float64(0.1592650520166779)}


Bert Score

In [30]:
from tqdm.auto import tqdm
from bert_score import score as bertscore
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

bert_f1_list = []

for pred, ref in tqdm(zip(preds, refs), total=len(preds), desc="DistilBERT BERTScore"):

    _, _, F1 = bertscore(
        [pred],
        [ref],
        lang="en",
        model_type="distilbert-base-uncased",
        device=device
    )

    bert_f1_list.append(float(F1[0]))

mean_f1 = sum(bert_f1_list) / len(bert_f1_list)

print("\nFinal Mean BERTScore F1:", mean_f1)


Using device: cuda


DistilBERT BERTScore:   0%|          | 0/150 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]


Final Mean BERTScore F1: 0.7516549543539683
